In [33]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM

drive.mount('/content/drive')
train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')

df = train.copy()

bash: line 15: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


In [34]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

bash: line 8: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


    max_len : 800
    topk : 1000
    min_count = 500

In [40]:


sparse_feat = [
    SparseFeat('gender', vocabulary_size= 2 , embedding_dim= 4),
    SparseFeat('age_group' , vocabulary_size= 8 , embedding_dim= 8),
    SparseFeat('inventory_id' , vocabulary_size= 16, embedding_dim= 8 , embedding_name = 'lala'),
    SparseFeat('day_of_week', vocabulary_size=7 , embedding_dim= 8),
    SparseFeat('hour', vocabulary_size=24 , embedding_dim=16),
    SparseFeat('l_feat_14', vocabulary_size= 1131, embedding_dim= 16 ),
]

ordinal_sparse = [
    SparseFeat('l_feat_3' ,vocabulary_size= 3 , embedding_dim= 4),
    SparseFeat('l_feat_27',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_e_4' ,vocabulary_size= 4 , embedding_dim= 4),
    SparseFeat('feat_a_1' ,vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_3' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_4' ,vocabulary_size= 6 , embedding_dim= 8),
    SparseFeat('feat_a_8' ,vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_13',vocabulary_size= 5 , embedding_dim= 4),
    SparseFeat('feat_a_16',vocabulary_size= 7 , embedding_dim= 8),
    SparseFeat('feat_a_18',vocabulary_size= 7 , embedding_dim= 8)
]

# ordinal_scores = [
#     DenseFeat('l_feat_3_ordscore', 1),
#     DenseFeat('feat_a_1_ordscore', 1),

varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size= 2715931 ,
                            embedding_dim= 32 ,
                            use_hash=True ,
                            embedding_name = 'lala'),
    maxlen = 150,
    combiner = 'mean',
    length_name= 'seq_len',
    weight_name = None,
    weight_norm = False
)


seq_col = 'seq'
label_col = 'clicked'


nominal_names = [f.name for f in sparse_feat]
ordinal_names = [f.name for f in ordinal_sparse]

all_cols = df.columns.tolist()
exclude = set(nominal_names + ordinal_names + [label_col, seq_col, seq_len_col])

dense_feats = [DenseFeat(c, 1) for c in all_cols if c not in exclude]


bash: line 62: warning: here-document at line 1 delimited by end-of-file (wanted `PY')


In [ ]:
if dense_feats:
    mms = MinMaxScaler()
    df[dense_feats] = mms.fit_transform(df[dense_feats])



In [ ]:
%%bash
cat > /tmp/train_deepfm.py <<'PY'
import os, runpy, numpy as np, pandas as pd
os.environ["TF_USE_LEGACY_KERAS"]="1"
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"   # TF 경고 로그 축소
runpy.run_path("/tmp/keras_private_shim.py", run_name="__main__")

# ________________________________________________________________________

dnn_feature_columns = varlen_seq
linear_feature_columns = sparse_feat + ordinal_sparse + dense_feats

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용
# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭
